In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [9]:
nombreTabla = "SalesReason"
esquema = "Sales"

dimensionSalesReason = pd.read_sql_table(nombreTabla, motorBaseDatos, esquema)
dimensionSalesReason.head()

c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)
c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'ReasonType'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,SalesReasonID,Name,ReasonType,ModifiedDate
0,1,Price,Other,2008-04-30
1,2,On Promotion,Promotion,2008-04-30
2,3,Magazine Advertisement,Marketing,2008-04-30
3,4,Television Advertisement,Marketing,2008-04-30
4,5,Manufacturer,Other,2008-04-30


TRANSFORMACION

In [11]:
dimensionSalesReason.rename(columns={
    'Name' : 'SalesReasonName',
    'ReasonType' : 'SalesReasonReasonType',
    'SalesReasonID': 'SalesReasonKey'
},inplace=True)

dimensionSalesReason['SalesReasonAlternateKey'] = dimensionSalesReason["SalesReasonKey"]

dimensionSalesReason.drop(columns={
    'ModifiedDate',
},inplace=True)

dimensionSalesReason.head()

,SalesReasonKey,SalesReasonName,SalesReasonReasonType,SalesReasonAlternateKey
0,1,Price,Other,1
1,2,On Promotion,Promotion,2
2,3,Magazine Advertisement,Marketing,3
3,4,Television Advertisement,Marketing,4
4,5,Manufacturer,Other,5


CARGAR A LA BODEGA

In [13]:
dimensionSalesReason.to_sql('dimensionSalesReason',motorBodegaDatos, if_exists='replace',index=False)

10